# 🎙️ Clonador de Voz Pocket TTS & Gerador de Áudio para o Anki

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dankkss/Anki-PocketTTS-Bot/blob/main/notebooks/Clonador_PocketTTS_Anki.ipynb)

Este caderno executa o modelo **Pocket TTS** (Kyutai) na nuvem do Google Colab com **12 GB de RAM e GPU T4 Gratuita**, sem limites de memória e sem travar seu servidor.

### 🚀 O que este caderno faz:
1. **Clonagem Zero-Shot:** Extrai o timbre de qualquer voz a partir de um áudio de 5 a 10 segundos.
2. **Geração Individual:** Sintetiza qualquer frase com a voz clonada e gera `.mp3` leve.
3. **Super Geração em Lote:** Cole uma lista de frases e gere todos os áudios do seu baralho Anki de uma só vez em um arquivo `.zip` para download!

## 1. Instalação de Dependências

In [ ]:
#@title Instalar Pocket TTS e ferramentas de áudio (Leva ~25s)
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install -q pocket-tts soundfile scipy pydub huggingface_hub
print('✅ Dependências instaladas com sucesso!')

## 2. Autenticação Hugging Face & Carregamento do Modelo
Como os pesos do `kyutai/pocket-tts` são sob licença aberta com acesso verificado, insira seu token do Hugging Face abaixo (você pode gerar um gratuito em [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)).

In [ ]:
#@title Carregar o Modelo Pocket TTS
import os
import torch
from huggingface_hub import login
from pocket_tts import TTSModel, export_model_state
import soundfile as sf
from IPython.display import Audio, display

HF_TOKEN = "" #@param {type:"string"}

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    login(token=HF_TOKEN.strip(), add_to_git_credential=False)

print(f"Dispositivo ativo: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
print("Carregando modelo Pocket TTS na memória do Colab (12 GB)... aguarde alguns segundos.")
model = TTSModel.load_model()
print("✅ Modelo Pocket TTS pronto para síntese!")

## 3. Clonar uma Voz a partir de um Áudio
Envie um arquivo de áudio (MP3, OGG, WAV ou mensagem de voz) com 5 a 10 segundos de fala limpa.

In [ ]:
#@title Enviar Áudio e Extrair Perfil (.safetensors)
import subprocess
from google.colab import files
import re

NOME_DA_VOZ = "minha_voz" #@param {type:"string"}

slug_nome = re.sub(r"[^\w-]", "_", NOME_DA_VOZ.strip().lower()) or "minha_voz"
print("Selecione o arquivo de áudio para upload:")
uploaded = files.upload()

if not uploaded:
    print("Nenhum arquivo enviado.")
else:
    input_filename = list(uploaded.keys())[0]
    clean_wav = f"prep_{slug_nome}.wav"
    safetensors_file = f"{slug_nome}.safetensors"
    
    # Normalizar para 16kHz mono WAV via ffmpeg
    subprocess.run([
        "ffmpeg", "-y", "-i", input_filename,
        "-ar", "16000", "-ac", "1", "-c:a", "pcm_s16le",
        clean_wav
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # Extrair voice state
    voice_state = model.get_state_for_audio_prompt(clean_wav)
    export_model_state(voice_state, safetensors_file)
    
    print(f"\n🎉 Sucesso! Perfil clonado e salvo como: {safetensors_file}")
    print("Você pode usar esta voz nas células abaixo.")

## 4. Testar Voz Clonada (Frase Única)
Digite qualquer texto para gerar o áudio `.mp3` imediatamente.

In [ ]:
#@title Gerar Áudio Individual
TEXTO = "Hello! This is a test of my cloned voice for Anki flashcards." #@param {type:"string"}

if 'voice_state' not in globals():
    print("⚠️ Por favor, clone uma voz na célula 3 primeiro.")
else:
    print("Sintetizando áudio...")
    audio = model.generate_audio(voice_state, TEXTO)
    
    wav_out = "temp_single.wav"
    mp3_out = f"anki_{slug_nome}_teste.mp3"
    sf.write(wav_out, audio.numpy(), model.sample_rate)
    
    # Converter para MP3 192k
    subprocess.run([
        "ffmpeg", "-y", "-i", wav_out,
        "-codec:a", "libmp3lame", "-b:a", "192k",
        mp3_out
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    display(Audio(mp3_out, autoplay=True))
    print(f"Arquivo gerado: {mp3_out}")
    files.download(mp3_out)

## 5. ⚡ Super Geração em Lote para o Anki (Várias Frases de Uma Vez)
Cole uma lista de frases (uma por linha). O script gera todos os arquivos `.mp3`, nomeia cada um organizadamente e baixa tudo compactado em `anki_cartoes.zip`!

In [ ]:
#@title Geração em Lote e Download em .ZIP
import zipfile
import time

LISTA_FRASES = """Good morning, how are you? 
I need to practice my pronunciation every single day.
Consistency is the key to mastering any new language.
See you tomorrow!""" #@param {type:"string"}

if 'voice_state' not in globals():
    print("⚠️ Por favor, execute a célula 3 primeiro para carregar ou clonar sua voz.")
else:
    lines = [l.strip() for l in LISTA_FRASES.splitlines() if l.strip()]
    print(f"Total de frases para processar: {len(lines)}")
    
    output_dir = Path("lote_anki_mp3")
    output_dir.mkdir(exist_ok=True)
    
    generated_files = []
    t0 = time.time()
    
    for idx, phrase in enumerate(lines, 1):
        clean_slug = re.sub(r"[^\w-]", "_", phrase[:25].strip().lower()) or f"card_{idx}"
        wav_file = output_dir / f"{idx:03d}_{clean_slug}.wav"
        mp3_file = output_dir / f"{idx:03d}_{clean_slug}.mp3"
        
        audio = model.generate_audio(voice_state, phrase)
        sf.write(str(wav_file), audio.numpy(), model.sample_rate)
        
        subprocess.run([
            "ffmpeg", "-y", "-i", str(wav_file),
            "-codec:a", "libmp3lame", "-b:a", "192k",
            str(mp3_file)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        if wav_file.exists():
            wav_file.unlink()
            
        generated_files.append(mp3_file)
        print(f"[{idx}/{len(lines)}] Gerado: {mp3_file.name}")
        
    zip_filename = "anki_cartoes_audio.zip"
    with zipfile.ZipFile(zip_filename, 'w') as zf:
        for f in generated_files:
            zf.write(f, arcname=f.name)
            
    dur = time.time() - t0
    print(f"\n🎉 Concluído em {dur:.1f}s! Baixando {zip_filename}...")
    files.download(zip_filename)